# NullVector — Full Ingestion Pipeline with PostgreSQL Storage

Demonstrates the complete ingestion pipeline backed by PostgreSQL:

```
PDF
 └─ Phase 1: Acquisition   (acquire_document)
     └─ Phase 2: Tree        (build_tree + LLM summarization)
         └─ Phase 3: Corpus  (RetrievalCorpusBuilder)
             └─ Phase 4: Enrich (enrich_visual_region per visual unit)
```

All artifacts are persisted to PostgreSQL via `PostgresStorageConfig`.
At the end, manifest refs are saved so the LangGraph QA notebook can pick up directly.

When `summarize=True` is enabled, tree summarization now groups LLM-needed nodes
per tree level through `GatewayService.invoke_many()` inside each document build.
`build_tree_batch()` itself still fan-outs documents across isolated workers, but
batch-fatal gateway auth/config failures now raise after worker completion instead
of being returned only as `BatchResult.failed`.

**Prerequisites:**
- `uv sync --extra dev --extra postgres`
- PostgreSQL reachable at the URI below
- An `OPENROUTER_API_KEY` env var (or substitute your provider below)
- A PDF file to ingest


In [1]:
import json
import os
import uuid
from collections import Counter
from pathlib import Path

from pydantic import BaseModel

from nullvector.domain.common import BatchResult
from nullvector.domain.ledger import AcquisitionRequest, AcquisitionSettings
from nullvector.domain.retrieval import RetrievalUnitType
from nullvector.domain.tree import TreeBuildRequest, VisualEnrichmentRequest
from nullvector.ingest.acquisition_service import acquire_batch
from nullvector.llm import (
    GatewayConfig,
    GatewayRequest,
    GatewayService,
    LLMMessage,
    LLMRole,
    NoopProviderAdapter,
    NoopScriptedResponse,
    enrich_visual_region,
)
from nullvector.llm.adapters import LiteLLMAdapter
from nullvector.retrieval.build import RetrievalCorpusBuilder
from nullvector.retrieval.enrichment import augment_corpus_with_attachments
from nullvector.retrieval.load import load_retrieval_corpus
from nullvector.storage.config import PostgresStorageConfig
from nullvector.tree.service import build_tree_batch


## Configuration

Set `PDF_PATH` to the PDF you want to ingest. Run IDs are generated fresh each run;
re-using the same ID against the same PDF is idempotent (the run is skipped and the
cached manifest returned).

In [2]:
POSTGRES_URI = "postgresql://REDACTED_DB_CRED@localhost:5432/app"
PDF_PATH = "/home/pruthvi/projects/NullVector/cookbook/903000608.pdf"  # ← replace with your PDF
ACQ_RUN_ID = f"acq-{uuid.uuid4().hex[:8]}"
TREE_RUN_ID = f"tree-{uuid.uuid4().hex[:8]}"
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "REDACTED_OPENROUTER_KEY")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "REDACTED_GROQ_KEY")

storage = PostgresStorageConfig(conninfo=POSTGRES_URI)

print(f"ACQ_RUN_ID  : {ACQ_RUN_ID}")
print(f"TREE_RUN_ID : {TREE_RUN_ID}")
print(f"Storage     : {storage.backend}")

ACQ_RUN_ID  : acq-acc1aac9
TREE_RUN_ID : tree-5d54d310
Storage     : postgres


## LLM Gateway Setup

`GatewayService` is the NullVector typed LLM client. It handles retries, structured
output validation, auditing, and ordered `invoke_many()` batching. Here we use LiteLLM
routing via OpenRouter so any model can be swapped by changing `model`.

The new batch facade is gateway-side: tree summarization groups LLM-needed nodes per
level within each document build, while provider adapters continue to execute normal
typed requests.


In [3]:
gateway_config = GatewayConfig(
    default_model="openrouter/google/gemini-3.1-flash-lite-preview",
    timeout_seconds=45.0,
)
gateway = GatewayService(gateway_config, provider_adapter=LiteLLMAdapter(api_key=OPENROUTER_API_KEY))
print("Gateway ready:", gateway_config.default_model)

Gateway ready: openrouter/google/gemini-3.1-flash-lite-preview


## Gateway Batch Facade Demo

This deterministic noop example shows the ordered `invoke_many()` surface
that tree summarization now uses internally. It is not provider-native
bulk submission and does not change `build_tree_batch()` document isolation.
If the shared gateway is globally broken, `build_tree_batch()` can still raise
after its workers complete instead of returning only per-item failures.


In [4]:
# deterministic gateway batch demo
class BatchEchoResponse(BaseModel):
    message: str

batch_demo_gateway = GatewayService(
    GatewayConfig(default_model="postgres-cookbook-noop"),
    provider_adapter=NoopProviderAdapter(
        {
            "postgres-batch-a": NoopScriptedResponse(
                output_json={"message": "tree-level batch item A"}
            ),
            "postgres-batch-b": NoopScriptedResponse(
                output_json={"message": "tree-level batch item B"}
            ),
        }
    ),
)

batch_demo_messages = [
    success.output.message
    for success in batch_demo_gateway.invoke_many(
        (
            GatewayRequest[BatchEchoResponse](
                operation_name="postgres-batch-a",
                messages=(LLMMessage(role=LLMRole.USER, content="batch item A"),),
                response_model=BatchEchoResponse,
                idempotency_key="postgres-batch-a",
            ),
            GatewayRequest[BatchEchoResponse](
                operation_name="postgres-batch-b",
                messages=(LLMMessage(role=LLMRole.USER, content="batch item B"),),
                response_model=BatchEchoResponse,
                idempotency_key="postgres-batch-b",
            ),
        ),
        max_workers=4,
    )
]

print(json.dumps({"ordered_messages": batch_demo_messages}, indent=2))


{
  "ordered_messages": [
    "tree-level batch item A",
    "tree-level batch item B"
  ]
}


## Phase 1 — Acquisition

`acquire_document` fingerprints the PDF, extracts all text blocks and visual regions
via PyMuPDF, builds the canonical document ledger, and persists everything to PostgreSQL.

Re-running with the same `acquisition_run_id` against the same PDF is **idempotent**:
the existing manifest is returned without re-extracting.

In [5]:
# acq_request = AcquisitionRequest(
#     source_path=PDF_PATH,
#     acquisition_run_id=ACQ_RUN_ID,
#     settings=AcquisitionSettings(),
# )

# acq_manifest = acquire_document(acq_request, storage=storage)

# print(f"Document ID     : {acq_manifest.document_id}")
# print(f"Run ID          : {acq_manifest.acquisition_run_id}")
# print(f"Page count      : {acq_manifest.page_count}")
# print(f"Outline source  : {acq_manifest.selected_outline_source}")
# print(f"Ledger ref      : {acq_manifest.ledger_path}")

## Phase 2 — Tree Pipeline with LLM Summarization

`build_tree` builds the deterministic document hierarchy: heading extraction,
TOC reconciliation, outline anchoring, and hierarchy synthesis.

With `summarize=True` and a `gateway`, each committed node receives an LLM-generated
summary stored as a `NodeSummary` artifact. LLM-needed nodes are summarized
level-by-level through `gateway.invoke_many()` inside each document build, which
improves retrieval quality without changing provider adapters or tree-run isolation.

**Note:** For PostgreSQL storage, `acquisition_manifest_path` must be a `pg://` reference.


In [6]:
# # Construct the PostgreSQL artifact reference for the acquisition manifest.
# # Format: pg://{run_type}/{run_id}/{document_id}/{artifact_path}
# acq_manifest_pg_ref = (
#     f"pg://acquisition/{acq_manifest.acquisition_run_id}"
#     f"/{acq_manifest.document_id}/manifest.json"
# )

# tree_request = TreeBuildRequest(
#     acquisition_manifest_path=acq_manifest_pg_ref,
#     tree_run_id=TREE_RUN_ID,
#     summarize=True,  # batches LLM-needed node summaries level-by-level via gateway.invoke_many()
# )
# tree_manifest = build_tree(tree_request, gateway=gateway, storage=storage)

# print(f"Tree run ID     : {tree_manifest.tree_run_id}")
# print(f"Committed nodes : {tree_manifest.committed_node_count}")
# print(f"Unassigned spans: {tree_manifest.unassigned_span_count}")
# print(f"Strategy        : {tree_manifest.settings}")

## Phase 3 — Build Retrieval Corpus

`RetrievalCorpusBuilder` synthesizes a flat `RetrievalCorpus` from acquisition +
tree artifacts. Each unit is typed (`page_text`, `node_text`, `table`, `visual`,
`unresolved_visual`, `node_summary`) and carries a trust tier.

## Batch Ingestion (Multiple PDFs)

`acquire_batch` and `build_tree_batch` process multiple documents concurrently
using a `ThreadPoolExecutor`. Each document still runs in an isolated worker.
Ordinary per-document failures are collected into `BatchResult.failed`, while
batch-fatal gateway auth/config failures are raised after worker completion.
When `summarize=True`, each worker now batches its own tree-level summary calls
through `GatewayService.invoke_many()`, but `build_tree_batch()` is still not one
provider-native bulk LLM request across documents.

Swap `BATCH_PDF_PATHS` for your own list of PDFs.


In [7]:
BATCH_PDF_PATHS = [
    PDF_PATH,
    # "/path/to/second.pdf",
    # "/path/to/third.pdf",
]

# Phase 1 batch: acquire all PDFs concurrently
acq_requests = [
    AcquisitionRequest(
        source_path=p,
        acquisition_run_id=f"acq-{uuid.uuid4().hex[:8]}",
        settings=AcquisitionSettings(),
    )
    for p in BATCH_PDF_PATHS
]

acq_batch: BatchResult = acquire_batch(acq_requests, storage=storage, max_workers=4)

print(f"Acquired  : {len(acq_batch.successful)} / {len(acq_requests)}")
for manifest in acq_batch.successful:
    print(f"  ✓ {manifest.document_id[:16]}…  pages={manifest.page_count}  run={manifest.acquisition_run_id}")
for failure in acq_batch.failed:
    print(f"  ✗ index={failure.item_index}  →  {failure.error_type}: {failure.error_message}")

# Phase 2 batch: build trees for all successfully acquired documents.
# Each worker still owns one document tree; summarize=True now batches
# LLM-needed node summaries level-by-level inside that document pipeline.
# Ordinary per-item failures come back in BatchResult.failed, but shared
# gateway auth/config failures now raise instead of returning only failures.
tree_requests = [
    TreeBuildRequest(
        acquisition_manifest_path=(
            f"pg://acquisition/{m.acquisition_run_id}/{m.document_id}/manifest.json"
        ),
        tree_run_id=f"tree-{uuid.uuid4().hex[:8]}",
        summarize=True,
    )
    for m in acq_batch.successful
]

tree_batch: BatchResult = build_tree_batch(
    tree_requests, storage=storage, gateway=gateway, max_workers=4
)

print(f"\nTree built: {len(tree_batch.successful)} / {len(tree_requests)}")
for manifest in tree_batch.successful:
    print(f"  ✓ {manifest.document_id[:16]}…  nodes={manifest.committed_node_count}  run={manifest.tree_run_id}")
for failure in tree_batch.failed:
    print(f"  ✗ index={failure.item_index}  →  {failure.error_type}: {failure.error_message}")


'utf-16-be' codec can't decode byte 0x31 in position 16: truncated data
initial string:b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.41'
'utf-16-be' codec can't decode byte 0x44 in position 16: truncated data
initial string:b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.cD'
Removed unexpected destination b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.cD' from destination
Removed unexpected destination b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.41' from destination
'utf-16-be' codec can't decode byte 0x31 in position 16: truncated data
initial string:b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.41'
'utf-16-be' codec can't decode byte 0x44 in position 16: truncated data
initial string:b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.cD'
Removed unexpected destination b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.cD' from destination
Removed unexpected destination b'\xfe\xff\x00_\x00P\x00A\x00G\x00E\x001.41' from destination


Acquired  : 1 / 1
  ✓ 798d2f27d45d2ccd…  pages=148  run=acq-7d514f5f


Provider List: https://docs.litellm.ai/docs/providers

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litell

KeyboardInterrupt: 


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers




Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.

In [13]:
tree_manifest_pg_ref = (
    f"pg://tree/{tree_manifest.tree_run_id}"
    f"/{tree_manifest.document_id}/manifest.json"
)

corpus_manifest = RetrievalCorpusBuilder(storage=storage).build(
    acquisition_manifest_path=acq_manifest_pg_ref,
    tree_manifest_path=tree_manifest_pg_ref,
)
corpus = load_retrieval_corpus(corpus_manifest.corpus_path, storage=storage)

print(f"Total units: {len(corpus.units)}")
for utype, count in sorted(Counter(u.unit_type for u in corpus.units).items()):
    print(f"  {utype:<25}: {count}")

NameError: name 'tree_manifest' is not defined

## Phase 4 — LLM Visual Enrichment

Visual units (`visual`, `unresolved_visual`) have no text initially.
`enrich_visual_region` sends each image through the LLM gateway to produce a
`VisualEnrichmentAttachment` with a summary, labels, and confidence score.

`augment_corpus_with_attachments` folds these enrichments back into the corpus
so visual units now have searchable text.

In [ ]:
VISUAL_PROMPT = (
    "Describe this image precisely. Include: "
    "(1) type — chart, diagram, photo, table, flowchart, or other; "
    "(2) key information conveyed; "
    "(3) any visible text, labels, or axis titles; "
    "(4) overall relevance to the surrounding document context."
)

visual_units = [
    u for u in corpus.units
    if u.unit_type in {RetrievalUnitType.VISUAL, RetrievalUnitType.UNRESOLVED_VISUAL}
    and u.visual_region is not None
]
print(f"Visual units to enrich: {len(visual_units)}")

attachments = []
for unit in visual_units:
    try:
        att = enrich_visual_region(
            gateway,
            VisualEnrichmentRequest(
                request_id=uuid.uuid4().hex,
                region=unit.visual_region,
                prompt=VISUAL_PROMPT,
                node_id=unit.visual_region.node_id,
            ),
        )
        attachments.append(att)
        region_short = unit.visual_region.region_id[:20]
        conf = f"{att.confidence:.2f}" if att.confidence is not None else "n/a"
        print(f"  ✓ {region_short}…  confidence={conf}  labels={list(att.insight.labels[:3])}")
    except Exception as exc:
        print(f"  ✗ {unit.visual_region.region_id[:20]}…  {exc}")

enriched_corpus = augment_corpus_with_attachments(
    corpus=corpus,
    attachments=tuple(attachments),
)
print(f"\nEnriched {len(attachments)}/{len(visual_units)} visual units")

## Inspect a Sample Visual Enrichment

Show what an enriched visual unit looks like before handing off to the QA agent.

In [ ]:
enriched_visual = [
    u for u in enriched_corpus.units
    if u.unit_type in {RetrievalUnitType.VISUAL, RetrievalUnitType.UNRESOLVED_VISUAL}
    and u.text
]
if enriched_visual:
    sample = enriched_visual[0]
    print(f"Unit type : {sample.unit_type}")
    print(f"Page span : {sample.page_span}")
    print(f"Text      : {sample.text[:400]}...")
else:
    print("No enriched visual units found.")

## Save Manifest Refs for the LangGraph Notebook

The next notebook (`04_nullvector_langgraph_qa_postgres.ipynb`) needs the corpus path
to load the retrieval corpus. We save all refs here.

In [ ]:
manifest_refs = {
    "acq_manifest_pg_ref": acq_manifest_pg_ref,
    "tree_manifest_pg_ref": tree_manifest_pg_ref,
    "corpus_path": corpus_manifest.corpus_path,
    "document_id": acq_manifest.document_id,
    "page_count": acq_manifest.page_count,
    "committed_node_count": tree_manifest.committed_node_count,
    "total_corpus_units": len(enriched_corpus.units),
    "enriched_visual_units": len(attachments),
}
refs_path = Path("/tmp/nullvector_manifest_refs.json")
refs_path.write_text(json.dumps(manifest_refs, indent=2))
print(f"Saved to: {refs_path}")
print(json.dumps(manifest_refs, indent=2))